In [ ]:
"""
evaluate.py

Evaluates the trained U-Net PINN on the test dataset.

Metrics:

    RMSE - Root Mean Squared Error
    MAE  - Mean Absolute Error
    R²   - Coefficient of Determination

The evaluation compares:

    predicted LST
        vs.
    target LST

Invalid/no-data pixels are excluded from the metrics
when a valid-pixel mask is available.
"""

from pathlib import Path

import numpy as np
import torch

from unet_model import UNet


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path(
    "/content/drive/MyDrive/SISTER/data"
)

PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"

VALID_MASK_PATH = (
    PROCESSED_DIR / "valid_mask.npy"
)


# ============================================================
# Configuration
# ============================================================

BATCH_SIZE = 2

TRAIN_RATIO = 0.70

VALIDATION_RATIO = 0.15

RANDOM_SEED = 42


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# Load processed data
# ============================================================

def load_processed_data():
    """
    Load processed model inputs and target LST.

    Expected shapes:

        X = (N, 10, 256, 256)

        Y = (N, 1, 256, 256)

    A valid-pixel mask is loaded when available.
    """

    X = np.load(
        PROCESSED_DIR / "X.npy"
    )

    Y = np.load(
        PROCESSED_DIR / "Y.npy"
    )


    # --------------------------------------------------------
    # Load valid-pixel mask if available
    # --------------------------------------------------------

    if VALID_MASK_PATH.exists():

        valid_mask = np.load(
            VALID_MASK_PATH
        )

        print(
            "Valid-pixel mask loaded:",
            valid_mask.shape
        )

    else:

        valid_mask = np.ones_like(
            Y,
            dtype=bool
        )

        print(
            "WARNING: No valid-pixel mask found."
        )

        print(
            "All pixels will be treated as valid."
        )


    return X, Y, valid_mask


# ============================================================
# Recreate the same test split used by dataset.py
# ============================================================

def get_test_data(
    X,
    Y,
    valid_mask
):
    """
    Reproduce the deterministic train/validation/test
    split used by dataset.py.
    """

    total_samples = len(X)

    rng = np.random.default_rng(
        RANDOM_SEED
    )

    indices = rng.permutation(
        total_samples
    )


    # --------------------------------------------------------
    # Calculate split sizes
    # --------------------------------------------------------

    train_size = int(
        total_samples * TRAIN_RATIO
    )

    validation_size = int(
        total_samples * VALIDATION_RATIO
    )


    # --------------------------------------------------------
    # Get test indices
    # --------------------------------------------------------

    test_indices = indices[
        train_size + validation_size:
    ]


    # --------------------------------------------------------
    # Extract test data
    # --------------------------------------------------------

    X_test = X[
        test_indices
    ]

    Y_test = Y[
        test_indices
    ]

    valid_mask_test = valid_mask[
        test_indices
    ]


    return (
        X_test,
        Y_test,
        valid_mask_test
    )


# ============================================================
# Calculate metrics
# ============================================================

def calculate_metrics(
    predictions,
    targets,
    valid_mask
):
    """
    Calculate RMSE, MAE, and R² using only valid pixels.

    Parameters
    ----------
    predictions : numpy.ndarray
        Predicted LST values.

    targets : numpy.ndarray
        Reference LST values.

    valid_mask : numpy.ndarray
        Boolean mask indicating valid target pixels.
    """

    predictions = predictions.astype(
        np.float64
    )

    targets = targets.astype(
        np.float64
    )

    valid_mask = valid_mask.astype(
        bool
    )


    # --------------------------------------------------------
    # Make sure mask shape matches target
    # --------------------------------------------------------

    if valid_mask.shape != targets.shape:

        raise ValueError(
            "Valid mask shape does not match "
            "target shape."
        )


    # --------------------------------------------------------
    # Select valid pixels only
    # --------------------------------------------------------

    valid_predictions = predictions[
        valid_mask
    ]

    valid_targets = targets[
        valid_mask
    ]


    if len(valid_targets) == 0:

        raise ValueError(
            "No valid pixels were found "
            "for evaluation."
        )


    # --------------------------------------------------------
    # Mean Squared Error
    # --------------------------------------------------------

    mse = np.mean(
        (
            valid_predictions
            -
            valid_targets
        ) ** 2
    )


    # --------------------------------------------------------
    # Root Mean Squared Error
    # --------------------------------------------------------

    rmse = np.sqrt(
        mse
    )


    # --------------------------------------------------------
    # Mean Absolute Error
    # --------------------------------------------------------

    mae = np.mean(
        np.abs(
            valid_predictions
            -
            valid_targets
        )
    )


    # --------------------------------------------------------
    # R²
    # --------------------------------------------------------

    ss_res = np.sum(
        (
            valid_targets
            -
            valid_predictions
        ) ** 2
    )

    ss_tot = np.sum(
        (
            valid_targets
            -
            np.mean(valid_targets)
        ) ** 2
    )


    if ss_tot == 0:

        r2 = 0.0

    else:

        r2 = 1.0 - (
            ss_res / ss_tot
        )


    return (
        rmse,
        mae,
        r2
    )


# ============================================================
# Load trained model
# ============================================================

def load_model():
    """
    Load the best trained U-Net model.
    """

    model = UNet(
        in_channels=10,
        out_channels=1
    ).to(
        DEVICE
    )


    model_path = (
        MODEL_DIR
        /
        "best_unet_model.pth"
    )


    if not model_path.exists():

        raise FileNotFoundError(
            "Trained model not found:\n"
            f"{model_path}"
        )


    model.load_state_dict(
        torch.load(
            model_path,
            map_location=DEVICE
        )
    )


    model.eval()


    return model


# ============================================================
# Generate test predictions
# ============================================================

def generate_predictions(
    model,
    X_test,
    batch_size=BATCH_SIZE
):
    """
    Generate predicted LST maps for the test inputs.
    """

    predictions = []


    with torch.no_grad():

        for start in range(
            0,
            len(X_test),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(X_test)
            )


            # ------------------------------------------------
            # Convert test batch to tensor
            # ------------------------------------------------

            inputs = torch.tensor(
                X_test[start:end],
                dtype=torch.float32,
                device=DEVICE
            )


            # ------------------------------------------------
            # U-Net prediction
            # ------------------------------------------------

            outputs = model(
                inputs
            )


            # ------------------------------------------------
            # Move prediction back to CPU
            # ------------------------------------------------

            predictions.append(
                outputs.cpu().numpy()
            )


    return np.concatenate(
        predictions,
        axis=0
    )


# ============================================================
# Evaluate model
# ============================================================

def evaluate_model():
    """
    Run the complete test evaluation.
    """

    print(
        "Using device:",
        DEVICE
    )


    # --------------------------------------------------------
    # Load processed data
    # --------------------------------------------------------

    (
        X,
        Y,
        valid_mask
    ) = load_processed_data()


    # --------------------------------------------------------
    # Check shapes
    # --------------------------------------------------------

    if X.ndim != 4:

        raise ValueError(
            f"X must be 4D, got {X.ndim}D."
        )


    if Y.ndim != 4:

        raise ValueError(
            f"Y must be 4D, got {Y.ndim}D."
        )


    if X.shape[1] != 10:

        raise ValueError(
            "Expected 10 input channels."
        )


    if Y.shape[1] != 1:

        raise ValueError(
            "Expected one target LST channel."
        )


    if len(X) != len(Y):

        raise ValueError(
            "X and Y must contain the same "
            "number of samples."
        )


    # --------------------------------------------------------
    # Get test split
    # --------------------------------------------------------

    (
        X_test,
        Y_test,
        valid_mask_test
    ) = get_test_data(
        X,
        Y,
        valid_mask
    )


    print(
        "\nTest input shape:",
        X_test.shape
    )

    print(
        "Test target shape:",
        Y_test.shape
    )

    print(
        "Test mask shape:",
        valid_mask_test.shape
    )


    # --------------------------------------------------------
    # Load trained model
    # --------------------------------------------------------

    model = load_model()


    # --------------------------------------------------------
    # Generate predictions
    # --------------------------------------------------------

    predictions = generate_predictions(
        model,
        X_test
    )


    print(
        "Prediction shape:",
        predictions.shape
    )


    # --------------------------------------------------------
    # Calculate metrics
    # --------------------------------------------------------

    (
        rmse,
        mae,
        r2
    ) = calculate_metrics(
        predictions,
        Y_test,
        valid_mask_test
    )


    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print(
        "\nTest Results"
    )

    print(
        "------------"
    )

    print(
        f"RMSE: {rmse:.4f}"
    )

    print(
        f"MAE:  {mae:.4f}"
    )

    print(
        f"R²:   {r2:.4f}"
    )


    # --------------------------------------------------------
    # Save predictions
    # --------------------------------------------------------

    prediction_path = (
        MODEL_DIR
        /
        "test_predictions.npy"
    )


    np.save(
        prediction_path,
        predictions
    )


    print(
        "\nPredictions saved to:"
    )

    print(
        prediction_path
    )


    # --------------------------------------------------------
    # Save test targets
    # --------------------------------------------------------

    target_path = (
        MODEL_DIR
        /
        "test_targets.npy"
    )


    np.save(
        target_path,
        Y_test
    )


    # --------------------------------------------------------
    # Save valid mask
    # --------------------------------------------------------

    mask_path = (
        MODEL_DIR
        /
        "test_valid_mask.npy"
    )


    np.save(
        mask_path,
        valid_mask_test
    )


    # --------------------------------------------------------
    # Save metrics
    # --------------------------------------------------------

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


    metrics_path = (
        MODEL_DIR
        /
        "evaluation_metrics.npy"
    )


    np.save(
        metrics_path,
        metrics,
        allow_pickle=True
    )


    print(
        "Metrics saved to:"
    )

    print(
        metrics_path
    )


    return (
        predictions,
        Y_test,
        valid_mask_test,
        metrics
    )


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    evaluate_model()